# Shift-Scan: Final Controlled Ablations & Results Consolidation
**Colab GPU notebook — research execution companion**

Project: *Power-of-Two Decay and Ternary Projections for Multiplication-Free State Space Language Models*

This notebook is a careful finish-work pipeline for the existing Shift-Scan project:
1. Mount Drive and recover the existing JSONL/CSV archive.
2. Preserve previous records (no overwriting / no fabricated results).
3. Define matched ablation protocol and seed plan.
4. Validate project scripts/checkpoints before launching.
5. Run your project's training/evaluation entrypoints after you explicitly map them.
6. Collect logs, metrics, and a paper-ready summary table.

**Important:** The recovered archive contains prior outcomes, not model checkpoints. A notebook cannot recreate missing checkpoints. Do not claim new experiments until their logs and metrics are produced. A single Colab session may not complete all 3 seeds and long-context tests; use the pilot token budget first, then scale only if runtime allows.

## 0. Runtime setup
In Colab: **Runtime → Change runtime type → GPU**. Prefer a T4/A100 if available. This cell checks the GPU; it does not silently install an unverified Mamba stack.

In [ ]:
import os, sys, json, csv, glob, time, subprocess, platform, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as e:
    print("Torch import issue:", repr(e))

## 1. Mount Google Drive and define paths
Put the recovered ZIP in Drive (or upload it in Colab) and set `ARCHIVE_PATH`. The project root should contain the actual model/training code and config. This notebook will not guess which script implements your experiment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT these paths for your Drive layout.
PROJECT_ROOT = Path('/content/drive/MyDrive/shift_scan_advanced_project')
ARCHIVE_PATH = Path('/content/drive/MyDrive/shift_scan_recovered_data.zip')
RUNS_DIR = PROJECT_ROOT / 'results' / 'final_controlled'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root exists:", PROJECT_ROOT.exists(), PROJECT_ROOT)
print("Recovered archive exists:", ARCHIVE_PATH.exists(), ARCHIVE_PATH)

## 2. Recover and inspect the prior dataset
This extracts the archive into a new folder, keeping originals untouched. We merge by appending only; duplicate IDs are not removed from the source archive.

In [ ]:
import zipfile
RECOVERED_DIR = Path('/content/shift_scan_recovered')
if ARCHIVE_PATH.exists():
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(RECOVERED_DIR)
    print("Extracted to", RECOVERED_DIR)
else:
    print("Archive missing. Upload/copy shift_scan_recovered_data.zip to ARCHIVE_PATH before continuing.")

runs_candidates = list(RECOVERED_DIR.rglob('runs.jsonl')) + list(PROJECT_ROOT.rglob('runs.jsonl'))
supp_candidates = list(RECOVERED_DIR.rglob('supplemental_metrics.jsonl'))
print("runs.jsonl candidates:", runs_candidates[:10])
print("supplemental candidates:", supp_candidates[:10])

## 3. Load recovered records and build a baseline snapshot
The `runs.jsonl` records are the canonical original run rows. Supplemental records may mix CSV-derived metrics and partial logs, so keep their `record_type` and `status` visible.

In [ ]:
def read_jsonl(path):
    rows=[]
    if path and Path(path).exists():
        with open(path, 'r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                line=line.strip()
                if not line: continue
                try: rows.append(json.loads(line))
                except Exception as e: print(f"Skipping malformed line {line_no}: {e}")
    return rows

RUNS_JSONL = runs_candidates[0] if runs_candidates else None
SUPP_JSONL = supp_candidates[0] if supp_candidates else None
recovered_runs = read_jsonl(RUNS_JSONL)
supplemental = read_jsonl(SUPP_JSONL)

print("Canonical recovered run rows:", len(recovered_runs))
print("Supplemental rows:", len(supplemental))
runs_df = pd.json_normalize(recovered_runs)
display(runs_df.head(10))

## 4. Existing reported results (context, not newly executed)
These values were summarized from your recovered records/log context. Token budgets differ; do **not** put them in one matched-seed comparison without labeling budget.

In [ ]:
historical = pd.DataFrame([
    {"experiment":"FP student", "tokens":"4M", "seed":"3-seed", "FineWeb PPL":"23.0868 / 23.1094 / 23.0948", "status":"Recovered"},
    {"experiment":"Ternary absmean", "tokens":"4M", "seed":"3-seed", "FineWeb PPL":"60.8882 / 70.9303 / 55.3509", "status":"Recovered"},
    {"experiment":"Ternary TWN", "tokens":"4M", "seed":"3-seed", "FineWeb PPL":"44.0985 / 43.2672 / 45.5476", "status":"Recovered"},
    {"experiment":"Ternary act-aware", "tokens":"4M", "seed":"3-seed", "FineWeb PPL":"48.5344 / 50.0394 / 50.4829", "status":"Recovered"},
    {"experiment":"v2_best", "tokens":"4M", "seed":"3-seed", "FineWeb PPL":"31.79395 / 31.76474 / 31.94346", "status":"Recovered"},
    {"experiment":"v2_best nearest", "tokens":"4M", "seed":"0", "FineWeb PPL":"31.81958", "status":"Recovered"},
    {"experiment":"v2_best dyadic2", "tokens":"4M", "seed":"0", "FineWeb PPL":"31.82718", "status":"Recovered"},
    {"experiment":"v2_best_long", "tokens":"12M", "seed":"0", "FineWeb PPL":"29.28271", "status":"Recovered"},
    {"experiment":"HARPQ", "tokens":"~20M", "seed":"0", "FineWeb PPL":"133.9073", "status":"Recovered; distinct protocol"},
])
display(historical)

## 5. Freeze the primary ablation matrix
The scientific question is the interaction between projection precision and decay parameterization. Use four cells. A0/A1 isolates projection precision under original decay; A2/A3 isolates decay under ternary projections (the matrix below is full factorial). Use identical tokenizer, initialization, data stream, sequence length, optimizer, schedule, token count, and evaluation.

In [ ]:
ABLATIONS = [
    dict(code='A0', projections='FP16', decay='original_FP', interpretation='reference baseline'),
    dict(code='A1', projections='ternary_TWN', decay='original_FP', interpretation='projection quantization only'),
    dict(code='A2', projections='FP16', decay='power_of_two', interpretation='decay parameterization only'),
    dict(code='A3', projections='ternary_TWN', decay='power_of_two', interpretation='combined method'),
]
SEEDS = [0, 1, 2]
# Quick controlled pilot to finish the first validation today. Increase only after this runs cleanly.
TARGET_TOKENS = 2_000_000
SEQ_LEN = 512
BATCH_SIZE = 1  # adjust to actual model and VRAM; use gradient accumulation in project config
HORIZON = 128
PROTOCOL_ID = f'matched_{TARGET_TOKENS//1_000_000}M_seq{SEQ_LEN}_seeds012_v1'

ablation_df = pd.DataFrame(ABLATIONS)
display(ablation_df)
print("Protocol:", PROTOCOL_ID)

## 6. Reproducibility helpers

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception as e: print("Torch seed setup warning:", e)

def git_info(root):
    root=Path(root)
    if not root.exists(): return {"root":str(root),"git_commit":None}
    try:
        commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'], text=True).strip()
    except Exception: commit=None
    return {"root":str(root),"git_commit":commit}

print(git_info(PROJECT_ROOT))

## 7. Verify actual code and checkpoint availability before runs
This searches for likely entrypoints but does not choose one automatically. Inspect output, then enter the real command in Section 8. The notebook must fail loudly rather than silently run the wrong script.

In [ ]:
if PROJECT_ROOT.exists():
    pyfiles = sorted(str(p.relative_to(PROJECT_ROOT)) for p in PROJECT_ROOT.rglob('*.py'))
    print("Python files (first 100):")
    print("\\n".join(pyfiles[:100]))
    ckpts = [p for p in PROJECT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.pt','.pth','.ckpt','.safetensors'}]
    print("\\nCheckpoint files:", len(ckpts))
    for p in ckpts[:30]: print(p)
else:
    print("PROJECT_ROOT does not exist. Set it to the project directory.")

## 8. Register the real training command (required user mapping)
The project-specific trainer/config names are not established by recovered metric rows alone. Edit the command template below to match the code you have. Use placeholders `{seed}`, `{ablation}`, `{tokens}`, `{seq_len}`, `{out}` only if your CLI supports them. Keep the run's stdout/stderr log.

In [ ]:
# EXAMPLE TEMPLATE ONLY — replace with your project's actual command.
# Do not run this unchanged. The command below is deliberately empty for safety.
COMMAND_TEMPLATE = ""

def make_command(ablation, seed, out_dir):
    if not COMMAND_TEMPLATE.strip():
        raise RuntimeError("Set COMMAND_TEMPLATE to the verified project training CLI first.")
    return COMMAND_TEMPLATE.format(
        seed=seed, ablation=ablation['code'], tokens=TARGET_TOKENS,
        seq_len=SEQ_LEN, out=str(out_dir)
    )

print("Training command registered:", bool(COMMAND_TEMPLATE.strip()))

## 9. Run matrix with a durable log and metadata record
Once the command template is confirmed, this loop executes the matrix. Set `RUN_EXPERIMENTS=True` only after reviewing the command. Logs and metadata are written per run. If one run fails, the loop records the return code and continues.

In [ ]:
RUN_EXPERIMENTS = False  # flip to True only after COMMAND_TEMPLATE validation
run_manifest = []

if RUN_EXPERIMENTS:
    for ab in ABLATIONS:
        for seed in SEEDS:
            run_id = f'{PROTOCOL_ID}_{ab["code"]}_seed{seed}'
            out_dir = RUNS_DIR / run_id
            out_dir.mkdir(parents=True, exist_ok=True)
            log_path = out_dir / 'stdout_stderr.log'
            set_seed(seed)
            cmd = make_command(ab, seed, out_dir)
            meta = {
                "run_id":run_id, "protocol_id":PROTOCOL_ID, "ablation":ab,
                "seed":seed, "target_tokens":TARGET_TOKENS, "seq_len":SEQ_LEN,
                "command":cmd, "started_unix":time.time(), **git_info(PROJECT_ROOT)
            }
            with open(out_dir/'run_metadata.json','w') as f: json.dump(meta,f,indent=2)
            print("\\n>>>", run_id, "\\n", cmd)
            with open(log_path,'w') as log:
                proc=subprocess.run(cmd, shell=True, cwd=str(PROJECT_ROOT),
                                   stdout=log, stderr=subprocess.STDOUT)
            meta["return_code"]=proc.returncode
            meta["finished_unix"]=time.time()
            with open(out_dir/'run_metadata.json','w') as f: json.dump(meta,f,indent=2)
            run_manifest.append(meta)
            print("Return code:", proc.returncode, "| log:", log_path)
else:
    print("Execution disabled. Review command mapping, then set RUN_EXPERIMENTS=True.")

## 10. Evaluation checklist and metrics schema
Each run should report: token count actually processed, validation FineWeb-Edu cross-entropy and perplexity, teacher PPL if distillation is part of the same protocol, state MSE (only if measured), numerical error / retention statistics, runtime, peak VRAM, and checkpoint path. Keep missing fields null; do not estimate them.

In [ ]:
METRIC_FIELDS = [
    "run_id","protocol_id","ablation","seed","tokens_actual","seq_len",
    "fineweb_eval_ce","fineweb_eval_ppl","teacher_eval_ppl","state_mse",
    "retention_error","runtime_seconds","peak_vram_gib","checkpoint_path",
    "status","notes"
]
metrics_path = RUNS_DIR / 'metrics_template.csv'
pd.DataFrame(columns=METRIC_FIELDS).to_csv(metrics_path,index=False)
print("Blank metrics template saved:", metrics_path)

## 11. Aggregate matched runs only
Fill `metrics_template.csv` from actual evaluator output. This section restricts aggregation to one `protocol_id` and same target budget; it shows mean, sample standard deviation, and seed count.

In [ ]:
metrics = pd.read_csv(metrics_path)
if len(metrics):
    metrics['fineweb_eval_ppl'] = pd.to_numeric(metrics['fineweb_eval_ppl'], errors='coerce')
    same_protocol = metrics[metrics['protocol_id'] == PROTOCOL_ID]
    summary = (same_protocol.groupby('ablation')['fineweb_eval_ppl']
               .agg(['count','mean','std']).reset_index())
    display(summary)
else:
    print("No evaluation rows yet. Add measured rows to", metrics_path)

## 12. Optional long-context stability follow-up
Do this only after matched short-context runs finish and checkpoints exist. Evaluate sequence lengths supported by the model's actual implementation (e.g. 512/1024/2048); record PPL by length, state norm, finite-value checks, and any instability. Do not change training budget or data for this evaluation.

In [ ]:
LONG_CONTEXT_LENGTHS = [512, 1024, 2048]
# Fill in only when evaluator CLI has been verified:
LONG_EVAL_COMMAND_TEMPLATE = ""
print("Long-context command configured:", bool(LONG_EVAL_COMMAND_TEMPLATE.strip()))

## 13. Numerical and result integrity checks

In [ ]:
def check_metrics_table(df):
    required={'run_id','protocol_id','ablation','seed','tokens_actual','fineweb_eval_ppl','status'}
    missing=required-set(df.columns)
    if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
    dupe=df['run_id'].duplicated().sum()
    print("Duplicate run_id rows:", int(dupe))
    print("Non-success rows:", int((df['status']!='success').sum()))
    print("Null PPL rows:", int(df['fineweb_eval_ppl'].isna().sum()))
    print("Token-count range:", df['tokens_actual'].min(), "to", df['tokens_actual'].max())
    return df

if len(metrics): check_metrics_table(metrics)

## 14. Export final tables and archive outputs
Exports include prior recovered records and newly logged run artifacts separately. This preserves provenance.

In [ ]:
EXPORT_DIR = Path('/content/shift_scan_final_export')
EXPORT_DIR.mkdir(exist_ok=True)
if RUNS_JSONL and Path(RUNS_JSONL).exists():
    shutil.copy2(RUNS_JSONL, EXPORT_DIR/'recovered_original_runs.jsonl')
if SUPP_JSONL and Path(SUPP_JSONL).exists():
    shutil.copy2(SUPP_JSONL, EXPORT_DIR/'recovered_supplemental_metrics.jsonl')
if metrics_path.exists():
    shutil.copy2(metrics_path, EXPORT_DIR/'new_experiment_metrics.csv')
if RUNS_DIR.exists():
    shutil.copytree(RUNS_DIR, EXPORT_DIR/'new_run_artifacts', dirs_exist_ok=True)
zip_out=Path('/content/shift_scan_final_export.zip')
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in EXPORT_DIR.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(EXPORT_DIR.parent))
print("Export archive:", zip_out)
print("Copy it to Drive before the Colab runtime disconnects.")

## 15. Paper-ready report scaffold (do not overclaim)
- **Primary result:** matched A0–A3 comparison at one token budget, with three seeds.
- **Decay ablation:** compare original FP decay vs power-of-two decay under matched projections.
- **Projection ablation:** FP16 vs ternary projections under matched decay.
- **Stability:** report only executed long-context evals and measured diagnostics.
- **Efficiency:** report measured latency / tokens-per-second / VRAM from optimized kernels. Analytical multiplication counts and a slower prototype kernel are not speedup evidence.
- **Limitations:** incomplete checkpoint recovery, finite token budget, architecture-specific result, hardware/kernel implementation constraints.

Before presenting: verify all rows have logs, actual token counts, config hashes, and evaluation provenance.